# RepFR Model Fitting - LohnasKahana2014

Fit multiple CMR variants to the Lohnas & Kahana 2014 free recall dataset.

**Paradigm:** Free Recall
**Reference:** @lohnas2014retrieved

In [ ]:
import os
import papermill as pm

from jaxcmr.helpers import find_project_root

## Dataset Configuration

In [ ]:
# ============================================================================
# DATASET CONFIGURATION - LohnasKahana2014
# ============================================================================
# Paradigm: Free Recall
# Reference: @lohnas2014retrieved
# ============================================================================

DATA_TAG = "LohnasKahana2014"
DATA_PATH = "data/LohnasKahana2014.h5"
TRIAL_QUERY = "data['list_type'] > 0"  # Exclude control lists for main fitting
CONTROL_QUERY = "data['list_type'] == 1"

In [ ]:
# ============================================================================
# BASE PARAMETERS (shared across all model fits)
# ============================================================================

project_root = find_project_root()
target_directory = os.path.join(project_root, "projects/repfr/results")
rendered_dir = os.path.join(project_root, "projects/repfr/notebooks/rendered")
templates_dir = os.path.join(project_root, "templates")

os.makedirs(rendered_dir, exist_ok=True)
os.makedirs(os.path.join(target_directory, "fits"), exist_ok=True)
os.makedirs(os.path.join(target_directory, "simulations"), exist_ok=True)
os.makedirs(os.path.join(target_directory, "figures/fitting"), exist_ok=True)

base_params = {
    # Flow control
    "redo_fits": True,
    "redo_sims": True,
    "redo_figures": True,
    "filter_repeated_recalls": True,
    "allow_repeated_recalls": False,
    
    # Run identification
    "run_tag": "full_best_of_3",
    "experiment_count": 50,
    
    # Data configuration
    "data_tag": DATA_TAG,
    "data_path": DATA_PATH,
    "trial_query": TRIAL_QUERY,
    "control_trial_query": CONTROL_QUERY,
    "target_dir": target_directory,
    
    # Fitting hyperparameters
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    
    # Analyses to run after fitting (from thesis)
    "single_analysis_paths": [
        "jaxcmr.analyses.repcrp.plot_rep_crp",
        "jaxcmr.analyses.backrepcrp.plot_back_rep_crp",
    ],
    "comparison_analysis_paths": [
        "jaxcmr.analyses.spc.plot_spc",
        "jaxcmr.analyses.crp.plot_crp",
        "jaxcmr.analyses.pnr.plot_pnr",
        "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_i2j",
        "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_j2i",
        "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_both",
        "jaxcmr.analyses.rpl.plot_rpl",
        "jaxcmr.analyses.rpl.plot_full_rpl",
    ],
}

## Model Configurations

Core models for Chapter 3 repetition analysis:
1. **WeirdCMR** - Standard composite context model (mfc_choice_sensitivity free)
2. **WeirdPositionalCMR** - Instance-based context representation (mfc_choice_sensitivity fixed to 1.0)
3. **WeirdNoReinstateCMR** - CMR without context reinstatement
4. **WeirdCMRDistinctContexts** - CMR with separate context representations

In [ ]:
# ============================================================================
# PARAMETER BOUNDS
# ============================================================================
# Note: NOT including stop_probability_scale/growth since using NoStopTermination
# ============================================================================

# Base free parameter bounds (shared by most models)
BASE_FREE_BOUNDS = {
    "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "shared_support": [2.220446049250313e-16, 100.0],
    "item_support": [2.220446049250313e-16, 100.0],
    "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
    "primacy_scale": [2.220446049250313e-16, 100.0],
    "primacy_decay": [2.220446049250313e-16, 100.0],
    "choice_sensitivity": [2.220446049250313e-16, 100.0],
}

# WeirdCMR: adds mfc_choice_sensitivity as FREE
WEIRD_CMR_FREE_BOUNDS = {
    **BASE_FREE_BOUNDS,
    "mfc_choice_sensitivity": [2.220446049250313e-16, 100.0],
}

# Model-specific configurations
varied_parameters = [
    # 1. WeirdCMR - mfc_choice_sensitivity is FREE
    {
        "model_name": "WeirdCMR",
        "model_factory_path": "jaxcmr.models_repfr.weird_cmr.BaseCMRFactory",
        "parameters": {
            "fixed": {},
            "free": WEIRD_CMR_FREE_BOUNDS,
        },
    },
    # 2. WeirdPositionalCMR - mfc_choice_sensitivity FIXED to 1.0
    {
        "model_name": "WeirdPositionalCMR",
        "model_factory_path": "jaxcmr.models_repfr.weird_positional_cmr.BaseCMRFactory",
        "parameters": {
            "fixed": {"mfc_choice_sensitivity": 1.0},
            "free": BASE_FREE_BOUNDS,
        },
    },
    # 3. WeirdNoReinstateCMR - no mfc_choice_sensitivity
    {
        "model_name": "WeirdNoReinstateCMR",
        "model_factory_path": "jaxcmr.models_repfr.weird_no_reinstate_cmr.BaseCMRFactory",
        "parameters": {
            "fixed": {},
            "free": BASE_FREE_BOUNDS,
        },
    },
    # 4. WeirdCMRDistinctContexts - no mfc_choice_sensitivity
    {
        "model_name": "WeirdCMRDistinctContexts",
        "model_factory_path": "jaxcmr.models_repfr.weird_cmr_distinct_contexts.BaseCMRFactory",
        "parameters": {
            "fixed": {},
            "free": BASE_FREE_BOUNDS,
        },
    },
]

## Run Model Fitting

In [ ]:
template_path = os.path.join(templates_dir, "fitting.ipynb")

for partial_params in varied_parameters:
    # Merge base and model-specific parameters
    params = base_params.copy()
    params.update(partial_params)
    
    # Construct unique output identifiers
    data_tag = params["data_tag"]
    model_name = params["model_name"]
    run_tag = params["run_tag"]
    
    # Output notebook path
    output_path = os.path.join(
        rendered_dir,
        f"fitting_{data_tag}_{model_name}_{run_tag}.ipynb"
    )
    
    print(f"Fitting {model_name} -> {output_path}")
    pm.execute_notebook(
        template_path,
        output_path,
        parameters=params,
        progress_bar=True,
    )